In [30]:
#!pip install pandas datasets scikit-learn

## Descriere dataset

Dataset-ul utilizat contine CV-uri provenite din surse publice disponibile online (HuggingFace dataset).

### Caracteristici:
- aproximativ 500 CV-uri utilizate pentru analiza
- datele sunt in format text (nestructurat)

### Coloane:
- resume_text – textul complet al CV-ului
- clean_resume – versiunea curatata a textului
- skills_found – lista de skill-uri extrase
- num_skills – numarul de skill-uri identificate

### Preprocesare aplicata:
- transformare text in lowercase
- eliminare caractere speciale
- eliminare spatii multiple
- extragere skill-uri prin keyword matching

### Limitari:
- unele CV-uri nu contin skill-uri detectabile
- lipsesc informatii personale (anonimizare)
- metoda de extragere este simpla (bazata pe cuvinte cheie)
- job descriptions sunt putine (doar cateva exemple demo)

In etapa urmatoare se va extinde dataset-ul si se vor folosi metode NLP mai avansate.

In [31]:
import re
import pandas as pd
from datasets import load_dataset
from collections import Counter

# 1. Incarcam CV-uri de pe HuggingFace
resumes_dataset = load_dataset("datasetmaster/resumes", split="train")

# transformam in DataFrame
resumes_df = resumes_dataset.to_pandas()

print(resumes_df.head())
print(resumes_df.columns)

                                       personal_info  \
0  {"name":"Unknown","email":"Unknown","phone":"U...   
1  {"name":"Unknown","email":"Unknown","phone":"U...   
2  {"name":"Not Provided","email":"Not Provided",...   
3  {"name":"Unknown","email":"Unknown","phone":"U...   
4  {"name":"","email":"","phone":"","location":{"...   

                                          experience  \
0  [{"company":"Fresher","company_info":{"industr...   
1  [{"company":"Delta Controls, Dubai FZCO","comp...   
2  [{"company":"Parkar Consulting and Labs","comp...   
3  [{"company":"Delta Controls, Dubai FZCO","comp...   
4  [{"company":"Atos Syntel","company_info":{"ind...   

                                           education  \
0  [{"degree":{"level":"ME","field":"Computer Eng...   
1  [{"degree":{"level":"B.E","field":"Electronics...   
2  [{"degree":{"level":"B.E.","field":"Not Provid...   
3  [{"degree":{"level":"B.E","field":"Electronics...   
4  [{"degree":{"level":"Bachelor of Engineerin

## Incarcarea datelor

In aceasta sectiune incarcam un dataset cu CV-uri folosind biblioteca HuggingFace Datasets. Datele sunt transformate intr-un DataFrame pentru a putea fi analizate si procesate mai usor.

In [32]:
# 2. Alegem coloana cu textul CV-ului
# Daca numele coloanei difera, afisarea de mai sus va arata ce trebuie schimbat

possible_resume_columns = ["resume", "text", "content", "Resume", "resume_text"]

resume_col = None
for col in possible_resume_columns:
    if col in resumes_df.columns:
        resume_col = col
        break

if resume_col is None:
    resume_col = resumes_df.columns[0]

print("Coloana folosita pentru CV:", resume_col)

resumes_df = resumes_df[[resume_col]].rename(columns={resume_col: "resume_text"})
resumes_df = resumes_df.dropna()
resumes_df = resumes_df.head(1000)

resumes_df.head()

Coloana folosita pentru CV: personal_info


,resume_text
0,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U..."
1,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U..."
2,"{""name"":""Not Provided"",""email"":""Not Provided"",..."
3,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U..."
4,"{""name"":"""",""email"":"""",""phone"":"""",""location"":{""..."


## Selectarea coloanei relevante

Dataset-ul poate avea mai multe coloane, dar pentru proiect ne intereseaza coloana care contine textul CV-ului. Aceasta este redenumita in `resume_text` pentru a fi mai usor de folosit in pasii urmatori.

In [33]:
# 3. Curatare text

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", " ", text)
    text = re.sub(r"[^a-zA-Z0-9+#. ]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

resumes_df["clean_resume"] = resumes_df["resume_text"].apply(clean_text)

resumes_df.head()

,resume_text,clean_resume
0,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U...",name unknown email unknown phone unknown locat...
1,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U...",name unknown email unknown phone unknown locat...
2,"{""name"":""Not Provided"",""email"":""Not Provided"",...",name not provided email not provided phone not...
3,"{""name"":""Unknown"",""email"":""Unknown"",""phone"":""U...",name unknown email unknown phone unknown locat...
4,"{""name"":"""",""email"":"""",""phone"":"""",""location"":{""...",name email phone location city pune country in...


## Curatarea textului

Textele din CV-uri sunt transformate in litere mici si sunt eliminate caracterele speciale inutile. Acest pas ajuta la standardizarea datelor si face mai usoara identificarea skill-urilor.

In [34]:
# 4. Lista de skill-uri cautate

skills_list = [
    "python", "java", "c++", "c#", "sql", "mysql", "postgresql",
    "excel", "power bi", "tableau",
    "machine learning", "deep learning", "nlp",
    "tensorflow", "pytorch", "scikit-learn",
    "html", "css", "javascript", "react",
    "git", "github", "docker", "linux",
    "communication", "leadership", "teamwork"
]

In [35]:
# 5. Extragere skill-uri din fiecare CV

def extract_skills(text, skills):
    found = []
    for skill in skills:
        if skill in text:
            found.append(skill)
    return found

resumes_df["skills_found"] = resumes_df["clean_resume"].apply(
    lambda x: extract_skills(x, skills_list)
)

resumes_df["num_skills"] = resumes_df["skills_found"].apply(len)

resumes_df[["clean_resume", "skills_found", "num_skills"]].head()

,clean_resume,skills_found,num_skills
0,name unknown email unknown phone unknown locat...,"[python, c++, sql, mysql, tensorflow, git, git...",7
1,name unknown email unknown phone unknown locat...,[],0
2,name not provided email not provided phone not...,"[git, github]",2
3,name unknown email unknown phone unknown locat...,[],0
4,name email phone location city pune country in...,"[git, github]",2


## Extragerea skill-urilor

Pentru fiecare CV, cautam aparitia unor skill-uri dintr-o lista predefinita. Daca un skill este gasit in text, acesta este salvat in lista de competente identificate pentru acel CV.

In [36]:
# 6. Analiza simpla a datelor

print("Numar CV-uri analizate:", len(resumes_df))
print("Lungime medie CV:", resumes_df["clean_resume"].apply(len).mean())
print("Numar mediu skill-uri gasite:", resumes_df["num_skills"].mean())

Numar CV-uri analizate: 1000
Lungime medie CV: 334.161
Numar mediu skill-uri gasite: 2.909


## Analiza datelor

In aceasta sectiune calculam statistici simple despre date: numarul de CV-uri analizate, lungimea medie a CV-urilor si numarul mediu de skill-uri gasite.

In [37]:
# 7. Top skill-uri gasite

all_skills = []

for skills in resumes_df["skills_found"]:
    all_skills.extend(skills)

skill_counts = Counter(all_skills)

top_skills_df = pd.DataFrame(
    skill_counts.items(),
    columns=["skill", "count"]
).sort_values(by="count", ascending=False)

top_skills_df.head(10)

,skill,count
6,github,982
5,git,982
7,java,239
0,python,234
16,react,200
10,javascript,106
17,docker,83
2,sql,15
11,machine learning,14
3,mysql,13


In [38]:
# 8. Job descriptions demo

job_descriptions = [
    "We are looking for a Python developer with SQL, Git and machine learning knowledge.",
    "Data analyst role requiring Excel, Power BI, SQL and communication skills.",
    "Frontend developer with HTML, CSS, JavaScript, React and Git experience.",
    "Backend developer with Java, Spring, SQL and Docker experience.",
    "Machine learning engineer with Python, TensorFlow, deep learning and NLP skills.",
    "DevOps engineer with Docker, Kubernetes, Linux and CI/CD experience.",
    "Software engineer with C++, algorithms, data structures and problem solving skills.",
    "Business analyst with Excel, communication, SQL and reporting experience"
]

jobs_df = pd.DataFrame({"job_description": job_descriptions})
jobs_df["clean_job"] = jobs_df["job_description"].apply(clean_text)
jobs_df["required_skills"] = jobs_df["clean_job"].apply(
    lambda x: extract_skills(x, skills_list)
)

jobs_df

,job_description,clean_job,required_skills
0,We are looking for a Python developer with SQL...,we are looking for a python developer with sql...,"[python, sql, machine learning, git]"
1,"Data analyst role requiring Excel, Power BI, S...",data analyst role requiring excel power bi sql...,"[sql, excel, power bi, communication]"
2,"Frontend developer with HTML, CSS, JavaScript,...",frontend developer with html css javascript re...,"[java, html, css, javascript, react, git]"
3,"Backend developer with Java, Spring, SQL and D...",backend developer with java spring sql and doc...,"[java, sql, docker]"
4,"Machine learning engineer with Python, TensorF...",machine learning engineer with python tensorfl...,"[python, machine learning, deep learning, nlp,..."
5,"DevOps engineer with Docker, Kubernetes, Linux...",devops engineer with docker kubernetes linux a...,"[docker, linux]"
6,"Software engineer with C++, algorithms, data s...",software engineer with c++ algorithms data str...,[c++]
7,"Business analyst with Excel, communication, SQ...",business analyst with excel communication sql ...,"[sql, excel, communication]"


## Job Descriptions

In aceasta etapa au fost utilizate mai multe descrieri de job (exemple sintetice) pentru a testa functionalitatea sistemului.

Acestea acopera diferite roluri:
- software developer
- data analyst
- machine learning engineer
- devops engineer

Limitari:
- job descriptions sunt generate manual (nu dataset real complet)
- nu acopera toate domeniile

In etapa urmatoare se va folosi un dataset mai complex si real pentru matching.

In [39]:
# 9. Salvam datele procesate

resumes_df.to_csv("processed_resumes.csv", index=False)
jobs_df.to_csv("processed_jobs.csv", index=False)

print("Fisiere salvate: processed_resumes.csv si processed_jobs.csv")

Fisiere salvate: processed_resumes.csv si processed_jobs.csv


## Observatii

- Unele CV-uri nu contin skill-uri detectabile din lista folosita.
- Unele informatii personale lipsesc sau sunt anonimizate.
- Datele sunt nestructurate si pot avea formate diferite.
- Pentru o extragere mai buna a informatiilor, in etapele urmatoare pot fi folosite metode NLP mai avansate.

In [40]:
# functie pentru calcul scor potrivire

def compute_match_score(cv_skills, job_skills):
    if len(job_skills) == 0:
        return 0

    matched = set(cv_skills) & set(job_skills)
    score = len(matched) / len(job_skills) * 100

    return round(score, 2), list(matched)

In [41]:
# luam primul CV din dataset
cv_example = resumes_df.iloc[0]

cv_skills = cv_example["skills_found"]

# exemplu job
job_skills = ["python", "sql", "docker", "git"]

score, matched = compute_match_score(cv_skills, job_skills)

missing = list(set(job_skills) - set(cv_skills))

print("Candidate Match Score:", score, "%")
print("Matched Skills:", matched)
print("Missing Skills:", missing)

Candidate Match Score: 75.0 %
Matched Skills: ['sql', 'python', 'git']
Missing Skills: ['docker']


## Exemplu output sistem

Pentru un candidat analizat:

- Candidate Match Score: procentul de skill-uri din job regasite in CV
- Matched Skills: skill-uri comune intre CV si job
- Missing Skills: skill-uri necesare dar absente

Acest tip de output ofera suport decizional pentru recrutori.

In [42]:
# test pe mai multi candidati

for i in range(3):
    cv = resumes_df.iloc[i]
    score, matched = compute_match_score(cv["skills_found"], job_skills)

    print("\nCandidat", i+1)
    print("Scor:", score, "%")
    print("Skill-uri potrivite:", matched)


Candidat 1
Scor: 75.0 %
Skill-uri potrivite: ['sql', 'python', 'git']

Candidat 2
Scor: 0.0 %
Skill-uri potrivite: []

Candidat 3
Scor: 25.0 %
Skill-uri potrivite: ['git']


## Matching automat pentru toate job descriptions

In aceasta sectiune testam sistemul pe mai multe descrieri de job. Pentru fiecare job, calculam scorul de potrivire pentru fiecare CV si afisam cei mai potriviti candidati.

In [43]:
matching_results = []

for job_index, job_row in jobs_df.iterrows():
    job_skills = job_row["required_skills"]

    for cv_index, cv_row in resumes_df.iterrows():
        cv_skills = cv_row["skills_found"]

        score, matched = compute_match_score(cv_skills, job_skills)

        # calculam missing separat
        missing = list(set(job_skills) - set(cv_skills))

        matching_results.append({
            "job_id": job_index,
            "candidate_id": cv_index,
            "job_description": job_row["job_description"],
            "match_score": score,
            "matched_skills": matched,
            "missing_skills": missing
        })

matching_df = pd.DataFrame(matching_results)

matching_df.head()

,job_id,candidate_id,job_description,match_score,matched_skills,missing_skills
0,0,0,We are looking for a Python developer with SQL...,75.0,"[sql, python, git]",[machine learning]
1,0,1,We are looking for a Python developer with SQL...,0.0,[],"[machine learning, sql, python, git]"
2,0,2,We are looking for a Python developer with SQL...,25.0,[git],"[machine learning, sql, python]"
3,0,3,We are looking for a Python developer with SQL...,0.0,[],"[machine learning, sql, python, git]"
4,0,4,We are looking for a Python developer with SQL...,25.0,[git],"[machine learning, sql, python]"


In [44]:
top_candidates = matching_df.sort_values(
    by=["job_id", "match_score"],
    ascending=[True, False]
).groupby("job_id").head(3)

top_candidates[[
    "job_id",
    "candidate_id",
    "match_score",
    "matched_skills",
    "missing_skills"
]]

,job_id,candidate_id,match_score,matched_skills,missing_skills
124,0,124,100.00,"[machine learning, sql, python, git]",[]
0,0,0,75.00,"[sql, python, git]",[machine learning]
26,0,26,75.00,"[sql, python, git]",[machine learning]
1151,1,151,50.00,"[communication, excel]","[power bi, sql]"
1162,1,162,50.00,"[communication, excel]","[power bi, sql]"
1163,1,163,50.00,"[communication, excel]","[power bi, sql]"
2007,2,7,83.33,"[java, css, html, javascript, git]",[react]
2035,2,35,83.33,"[java, css, html, javascript, git]",[react]
2066,2,66,83.33,"[java, css, html, javascript, git]",[react]
3007,3,7,66.67,"[java, sql]",[docker]


In [45]:
for job_id in jobs_df.index:
    print("\n==============================")
    print("JOB", job_id)
    print(jobs_df.loc[job_id, "job_description"])
    print("==============================")

    top = top_candidates[top_candidates["job_id"] == job_id]

    for _, row in top.iterrows():
        print("\nCandidat:", row["candidate_id"])
        print("Candidate Match Score:", row["match_score"], "%")
        print("Matched Skills:", row["matched_skills"])
        print("Missing Skills:", row["missing_skills"])


JOB 0
We are looking for a Python developer with SQL, Git and machine learning knowledge.

Candidat: 124
Candidate Match Score: 100.0 %
Matched Skills: ['machine learning', 'sql', 'python', 'git']
Missing Skills: []

Candidat: 0
Candidate Match Score: 75.0 %
Matched Skills: ['sql', 'python', 'git']
Missing Skills: ['machine learning']

Candidat: 26
Candidate Match Score: 75.0 %
Matched Skills: ['sql', 'python', 'git']
Missing Skills: ['machine learning']

JOB 1
Data analyst role requiring Excel, Power BI, SQL and communication skills.

Candidat: 151
Candidate Match Score: 50.0 %
Matched Skills: ['communication', 'excel']
Missing Skills: ['power bi', 'sql']

Candidat: 162
Candidate Match Score: 50.0 %
Matched Skills: ['communication', 'excel']
Missing Skills: ['power bi', 'sql']

Candidat: 163
Candidate Match Score: 50.0 %
Matched Skills: ['communication', 'excel']
Missing Skills: ['power bi', 'sql']

JOB 2
Frontend developer with HTML, CSS, JavaScript, React and Git experience.

Candi

## Matching CV - Job Description

Pentru fiecare CV si fiecare job description se calculeaza un scor de potrivire.

Scorul reprezinta procentul de skill-uri cerute de job care sunt gasite in CV.

- Matched Skills: skill-uri comune intre CV si job
- Missing Skills: skill-uri necesare dar absente din CV

Aceasta metoda este una simpla, bazata pe keyword matching, si va fi imbunatatita in etapele urmatoare folosind tehnici NLP mai avansate.

In [46]:
def generate_matching_results(resumes_df, jobs_df, output_file="matching_results.csv"):

    matching_results = []

    for job_index, job_row in jobs_df.iterrows():
        job_skills = job_row["required_skills"]

        for cv_index, cv_row in resumes_df.iterrows():
            cv_skills = cv_row["skills_found"]

            score, matched = compute_match_score(cv_skills, job_skills)

            # calculam missing skills
            missing = list(set(job_skills) - set(cv_skills))

            matching_results.append({
                "job_id": job_index,
                "candidate_id": cv_index,
                "job_description": job_row["job_description"],
                "match_score": score,
                "matched_skills": matched,
                "missing_skills": missing
            })

    matching_df = pd.DataFrame(matching_results)

    # salvare CSV
    matching_df.to_csv(output_file, index=False)

    print(f"Fisier salvat: {output_file}")

    return matching_df

In [47]:
matching_df = generate_matching_results(resumes_df, jobs_df)

Fisier salvat: matching_results.csv
